# LLM-as-a-Judge with RunPod-Hosted Models

This notebook judges OSQ responses using open-source models hosted on RunPod.

**Architecture:**
- Provision N RunPod pods with GPU
- Each pod hosts the same judge model (e.g., gpt-oss:120b) via Ollama
- Distribute evaluated models across pods
- Each pod judges its assigned models sequentially
- Download results to Phase 5 directory

**Key Features:**
- Pod-level parallelism (no within-pod threading)
- Append-only resumable design
- Phase alignment checks
- Compatible with existing llm-judge pipeline

# Configuration

In [2]:
import os
import time
import json
import stat
import posixpath
import runpod
import paramiko
import requests
from pathlib import Path
from datetime import datetime
from copy import deepcopy
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ───────────────────────────────────────────────
# RUNPOD CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]
runpod.api_key = RUNPOD_API_KEY

IMAGE_NAME = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
GPU_TYPE = "NVIDIA H200"  # Use H200 for large models like gpt-oss:120b
GPU_COUNT = 1
CONTAINER_DISK_GB = 200

# ───────────────────────────────────────────────
# JUDGE CONFIGURATION
# ───────────────────────────────────────────────
TASK_NAME = "sysengbench-osq"
JUDGE_MODEL = "gpt-oss:120b"  # Ollama model to use for judging
PROMPT_ID = "p1"  # "p1" for scores only, "p2" for scores + justifications
TEMPERATURE = 0.0
MAX_TOKENS = 2000
SAMPLE_N = 10  # 0 = judge ALL samples; else judge first N (for testing)

# ───────────────────────────────────────────────
# PARALLELISM
# ───────────────────────────────────────────────
MAX_CONCURRENT_PODS = 2  # Number of pods to run in parallel

# ───────────────────────────────────────────────
# PATHS
# ───────────────────────────────────────────────
BASE_DIR = Path.cwd()
PHASE4_ROOT = BASE_DIR / "../phase4_inference/output" / TASK_NAME
# PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge"
PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge-TEST" # use this one to dump files in a different dir/ for troubleshooting.
LOG_DIR = BASE_DIR / "runpod_llm_judge_logs"

PHASE5_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SSH_KEY_PATH = os.path.expanduser("~/.ssh/id_ed25519")

print(f"PHASE4_ROOT: {PHASE4_ROOT}")
print(f"PHASE5_ROOT: {PHASE5_ROOT}")
print(f"LOG_DIR: {LOG_DIR}")
print(f"RunPod API Key present: {bool(RUNPOD_API_KEY)}")

PHASE4_ROOT: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\..\phase4_inference\output\sysengbench-osq
PHASE5_ROOT: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\sysengbench-osq-llm-judge-TEST
LOG_DIR: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs
RunPod API Key present: True


# Judge Prompts

In [6]:
# Prompt 1: Scores only
JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>}},
  "conceptual_understanding": {{"score": <0–20>}},
  "completeness": {{"score": <0–20>}},
  "clarity_organization": {{"score": <0–20>}},
  "professional_relevance": {{"score": <0–20>}},
  "overall_score": <0–100>
}}
"""

# Prompt 2: Scores + justifications
JUDGE_PROMPT_P2 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# Select prompt based on PROMPT_ID
JUDGE_PROMPT = JUDGE_PROMPT_P1 if PROMPT_ID == "p1" else JUDGE_PROMPT_P2

# Helper Functions

In [7]:
def newest(path_iter):
    """Return the most recently modified file from an iterator."""
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    """Load all lines from a JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    """Safely parse JSON, return None on failure."""
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    """Extract student response from Phase 4 sample row."""
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    """Extract fields needed for judge prompt from Phase 4 row."""
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level = doc.get("blooms_level", "N/A")
    se_domain = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    """Check if required fields are missing."""
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    """Verify Phase 4 data hasn't changed since judging started."""
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} has no 'phase4_row' snapshot."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 content changed.\n"
            "Refusing to proceed. Freeze Phase-4 or regenerate Phase-5 from scratch."
        )

# RunPod Helper Functions

In [8]:
def run_and_check(ssh, cmd, desc, log_func):
    """Run a remote command and wait for completion."""
    log_func(f"▶ {desc}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    exit_code = stdout.channel.recv_exit_status()
    out = stdout.read().decode()
    err = stderr.read().decode()
    if out:
        log_func(out)
    if err:
        log_func(f"stderr: {err}")
    if exit_code != 0:
        raise RuntimeError(f"{desc} failed with exit code {exit_code}")
    log_func(f"✔ {desc} finished.")
    return out

def download_dir(sftp, remote_dir, local_dir, log_func):
    """Recursively download a directory from remote pod."""
    try:
        entries = sftp.listdir_attr(remote_dir)
    except FileNotFoundError:
        log_func(f"⚠ No output directory found at {remote_dir}")
        return
    os.makedirs(local_dir, exist_ok=True)
    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)
        local_path = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            download_dir(sftp, remote_path, local_path, log_func)
        else:
            sftp.get(remote_path, local_path)
            log_func(f"  ↓ {local_path}")

In [9]:
def create_pod_with_retries(log_func, max_attempts=3, retry_delay=10, **kwargs):
    """
    Attempt to create a RunPod pod with retries.
    Returns the created pod dict.
    Raises RuntimeError after max failures.
    """
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            log_func(f"Attempt {attempt}/{max_attempts}: creating pod...")
            pod = runpod.create_pod(**kwargs)
            pod_id = pod.get("id")
            log_func(f"Pod created: {pod_id}")
            return pod
        except Exception as e:
            last_err = e
            log_func(f"⚠ Pod creation failed on attempt {attempt}: {e}")
            if attempt < max_attempts:
                log_func(f"Waiting {retry_delay} seconds before retrying...")
                time.sleep(retry_delay)

    raise RuntimeError(f"❌ Failed to create pod after {max_attempts} attempts. Last error: {last_err}")


# Discover Models to Judge

In [10]:
import pandas as pd
import re

def build_multi_judge_progress_matrix(
    phase4_root: Path,
    phase5_root: Path,
    task_name: str,
    judge_model: str,
    prompt_id: str
) -> pd.DataFrame:
    """Build progress matrix to identify which models need judging."""
    p4 = phase4_root.parent / task_name
    p5 = phase5_root

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {p4}")

    model_folders = sorted([d.name for d in p4.iterdir() if d.is_dir()])
    judge_suffix = judge_model.replace(":", "_")
    target_pattern = re.compile(rf"__{judge_suffix}-{prompt_id}\.jsonl$")

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / model
        model_p5_dir = p5 / model

        # Total samples from Phase 4
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            sf = sample_files[0]
            with open(sf, "r", encoding="utf-8") as fh:
                total_samples = sum(1 for _ in fh)

        # Check for existing judgments
        judged_count = 0
        if model_p5_dir.exists():
            for jf in model_p5_dir.iterdir():
                if jf.is_file() and target_pattern.search(jf.name):
                    with open(jf, "r", encoding="utf-8") as fh:
                        judged_count = sum(1 for _ in fh)
                    break

        if total_samples == 0:
            status = "no_samples"
        elif judged_count == 0:
            status = "not_started"
        elif judged_count < total_samples:
            status = "partial"
        else:
            status = "complete"

        rows.append({
            "model_name": model,
            "ollama_name": model.replace("__", ":"),
            "judged_count": judged_count,
            "total_samples": total_samples,
            "progress_fraction": judged_count / total_samples if total_samples > 0 else 0.0,
            "status": status,
        })

    df = pd.DataFrame(rows)
    return df.sort_values("model_name")

# Build progress matrix
progress_df = build_multi_judge_progress_matrix(
    PHASE4_ROOT,
    PHASE5_ROOT,
    TASK_NAME,
    JUDGE_MODEL,
    PROMPT_ID
)

print("\n=== Judge Progress Matrix ===")
display(progress_df)

# Extract models that need judging
models_to_judge = progress_df[
    progress_df["status"].isin(["not_started", "partial"])
]["model_name"].tolist()

print(f"\nModels needing judgment: {len(models_to_judge)}")
print(models_to_judge)

utils.py            :164  2025-12-02 01:46:46,277 NumExpr defaulting to 8 threads.



=== Judge Progress Matrix ===


,model_name,ollama_name,judged_count,total_samples,progress_fraction,status
0,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,0,845,0.0,not_started
1,devstral__24b,devstral:24b,0,845,0.0,not_started
2,gemma3__12b,gemma3:12b,0,845,0.0,not_started
3,gemma3__1b,gemma3:1b,0,845,0.0,not_started
4,gemma3__27b,gemma3:27b,0,845,0.0,not_started
5,gemma3__4b,gemma3:4b,0,845,0.0,not_started
6,google__gemini-2.5-flash,google:gemini-2.5-flash,0,845,0.0,not_started
7,llama3.2__1b,llama3.2:1b,0,845,0.0,not_started
8,llama3.2__3b,llama3.2:3b,0,845,0.0,not_started
9,llama3.3__70b,llama3.3:70b,0,845,0.0,not_started



Models needing judgment: 19
['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b', 'llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']


# Pod Worker Function

This function runs on a single RunPod pod and judges all assigned models.

## Trying a more verbose version to understand what judgement progress is

still hanging before pulling a model. No progress outputs to pull a model...

In [34]:
POD_WORKER_SCRIPT_VERBOSE = r'''import json
import requests
from datetime import datetime
import sys

JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{
  "technical_accuracy": {"score": <0–20>},
  "conceptual_understanding": {"score": <0–20>},
  "completeness": {"score": <0–20>},
  "clarity_organization": {"score": <0–20>},
  "professional_relevance": {"score": <0–20>},
  "overall_score": <0–100>
}
"""

JUDGE_PROMPT_P2 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{
  "technical_accuracy": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "conceptual_understanding": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "completeness": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "clarity_organization": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "professional_relevance": {"score": <0–20>, "justification": "<1–2 sentences>"},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}
"""


def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None


def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""


def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level = doc.get("blooms_level", "N/A")
    se_domain = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }


def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]


def run_worker(input_path, output_path, judge_model, prompt_id,
               temperature, max_tokens, task_name, model_name):

    judge_prompt = JUDGE_PROMPT_P1 if prompt_id == "p1" else JUDGE_PROMPT_P2
    judge_url = "http://localhost:11434/v1/chat/completions"

    # count samples
    total_lines = sum(1 for _ in open(input_path, "r", encoding="utf-8"))
    print(f"[worker] Total OSQ samples: {total_lines}", flush=True)
    current = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            current += 1
            print(f"[worker] Progress: {current}/{total_lines}", flush=True)

            payload = json.loads(line)
            sid = payload["sample_id"]
            phase4_row = payload["phase4_row"]

            fields_for_prompt = derive_prompt_fields(phase4_row)
            missing = triad_missing(fields_for_prompt)

            if missing:
                record = {
                    "sample_id": sid,
                    "phase4_row": phase4_row,
                    "judge": {
                        "fields": None,
                        "prompt": None,
                        "raw_output": None,
                        "error": f"missing_fields: {missing}",
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": judge_model,
                            "temperature": temperature,
                            "max_tokens": max_tokens,
                            "task_name": task_name,
                            "model_name": model_name,
                            "prompt_id": prompt_id,
                        },
                    },
                }
                fout.write(json.dumps(record) + "\n")
                fout.flush()
                continue

            prompt = judge_prompt
            for key, val in fields_for_prompt.items():
                prompt = prompt.replace(f"{{{key}}}", val)

            raw = None
            parsed = None
            error = None

            try:
                response = requests.post(
                    judge_url,
                    headers={"Content-Type": "application/json"},
                    json={
                        "model": judge_model,
                        "messages": [
                            {
                                "role": "system",
                                "content": (
                                    "You are an expert evaluator in systems engineering education."
                                ),
                            },
                            {"role": "user", "content": prompt},
                        ],
                        "temperature": temperature,
                        "max_tokens": max_tokens,
                    },
                    timeout=200,
                )
                data = response.json()
                raw = data["choices"][0]["message"]["content"]
                parsed = safe_json(raw)
            except Exception as e:
                error = str(e)

            # Build output record
            if prompt_id == "p1":
                fields = {
                    "technical_accuracy": {"score": None},
                    "conceptual_understanding": {"score": None},
                    "completeness": {"score": None},
                    "clarity_organization": {"score": None},
                    "professional_relevance": {"score": None},
                    "overall_score": None,
                }
                if parsed:
                    fields["technical_accuracy"]["score"] = (parsed.get("technical_accuracy") or {}).get("score")
                    fields["conceptual_understanding"]["score"] = (parsed.get("conceptual_understanding") or {}).get("score")
                    fields["completeness"]["score"] = (parsed.get("completeness") or {}).get("score")
                    fields["clarity_organization"]["score"] = (parsed.get("clarity_organization") or {}).get("score")
                    fields["professional_relevance"]["score"] = (parsed.get("professional_relevance") or {}).get("score")
                    fields["overall_score"] = parsed.get("overall_score")
            else:
                fields = {
                    "technical_accuracy": {"score": None, "justification": None},
                    "conceptual_understanding": {"score": None, "justification": None},
                    "completeness": {"score": None, "justification": None},
                    "clarity_organization": {"score": None, "justification": None},
                    "professional_relevance": {"score": None, "justification": None},
                    "overall_score": None,
                    "overall_assessment": None,
                    "key_strengths": None,
                    "improvement_areas": None,
                }
                if parsed:
                    for key in ["technical_accuracy","conceptual_understanding","completeness","clarity_organization","professional_relevance"]:
                        fields[key] = {
                            "score": (parsed.get(key) or {}).get("score"),
                            "justification": (parsed.get(key) or {}).get("justification"),
                        }
                    fields["overall_score"] = parsed.get("overall_score")
                    fields["overall_assessment"] = parsed.get("overall_assessment")
                    fields["key_strengths"] = parsed.get("key_strengths")
                    fields["improvement_areas"] = parsed.get("improvement_areas")

            record = {
                "sample_id": sid,
                "phase4_row": phase4_row,
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "error": error,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": judge_model,
                        "temperature": temperature,
                        "max_tokens": max_tokens,
                        "task_name": task_name,
                        "model_name": model_name,
                        "prompt_id": prompt_id,
                    },
                },
            }
            fout.write(json.dumps(record) + "\n")
            fout.flush()


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--judge-model", required=True)
    parser.add_argument("--prompt-id", required=True)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--max-tokens", type=int, default=2000)
    parser.add_argument("--task-name", required=True)
    parser.add_argument("--model-name", required=True)
    args = parser.parse_args()

    run_worker(
        input_path=args.input,
        output_path=args.output,
        judge_model=args.judge_model,
        prompt_id=args.prompt_id,
        temperature=args.temperature,
        max_tokens=args.max_tokens,
        task_name=args.task_name,
        model_name=args.model_name,
    )
'''


---

# 🤖 Automated Workflow

This section provides **fully automated** orchestration of the LLM-as-a-judge workflow.

## Why Use Automation?

The **Manual Debugging Workflow** (below) is excellent for:
- Understanding how the system works
- Debugging connection issues
- Testing individual components
- Iterating on worker scripts

The **Automated Workflow** is better for:
- Production runs with many samples
- Batch processing multiple models
- Hands-off execution
- Consistent, repeatable results

## Features

- ✅ **Fully automated**: One function call executes all stages
- ✅ **Robust error handling**: Retries, timeouts, graceful failures
- ✅ **Real-time progress**: Streams pod output live
- ✅ **Auto cleanup**: Terminates pods even on errors
- ✅ **Comprehensive logging**: All actions logged to file
- ✅ **Cost optimization**: Immediate termination on completion

## How It Works

The automation module (`llm_judge_automation.py`) implements 5 stages:

1. **Create & Connect**: Provision RunPod, wait for SSH, establish connection
2. **Provision Environment**: Install Ollama, pull model, verify setup
3. **Deploy Script & Data**: Upload worker script and input samples
4. **Execute & Monitor**: Run worker with real-time progress streaming
5. **Cleanup**: Close SSH, terminate pod, generate summary

Each stage includes retry logic, progress tracking, and detailed logging.


In [ ]:
# Import the automation module
from llm_judge_automation import judge_with_automated_workflow

print("✓ Automation module loaded")
print("  Use judge_with_automated_workflow() for fully automated runs")

## 📘 Basic Usage

The simplest way to run an automated workflow:

```python
# 1. Load your input samples
samples = load_samples_from_phase4("anthropic__claude-sonnet-4.5")

# 2. Run automated workflow
result = judge_with_automated_workflow(
    task_name=TASK_NAME,
    model_name="anthropic__claude-sonnet-4.5",
    input_samples=samples[:10],  # Start with 10 for testing
    judge_model=JUDGE_MODEL,
    worker_script=POD_WORKER_SCRIPT_VERBOSE,
    log_dir=LOG_DIR,
)

# 3. Check results
if result['success']:
    print(f"✅ Success! Processed {result['samples_processed']} samples")
    print(f"   Output: {result['output_file']}")
    print(f"   Duration: {result['duration_seconds']:.1f}s")
else:
    print(f"❌ Failed: {result['error']}")
```

That's it! The function handles everything automatically.

In [ ]:
# Example: Run automated workflow on a small test batch
#
# UNCOMMENT AND RUN THIS CELL TO EXECUTE AN AUTOMATED WORKFLOW
#
# NOTE: This will create a RunPod instance and incur costs!

# # Select a test model
# test_model = "anthropic__claude-sonnet-4.5"
#
# # Load samples from Phase 4 output
# model_dir = PHASE4_ROOT.parent / TASK_NAME / test_model
# sample_file = newest(model_dir.glob("samples_*.jsonl"))
#
# print(f"Loading samples from: {sample_file}")
# with open(sample_file) as f:
#     all_samples = [json.loads(line) for line in f]
#
# # Test with just 3 samples first
# test_samples = all_samples[:3]
# print(f"Selected {len(test_samples)} samples for testing\n")
#
# # Run automated workflow
# result = judge_with_automated_workflow(
#     task_name=TASK_NAME,
#     model_name=test_model,
#     input_samples=test_samples,
#     judge_model=JUDGE_MODEL,
#     worker_script=POD_WORKER_SCRIPT_VERBOSE,
#     log_dir=LOG_DIR,
#     prompt_id=PROMPT_ID,
#     temperature=TEMPERATURE,
#     max_tokens=MAX_TOKENS,
#     image_name=IMAGE_NAME,
#     gpu_type=GPU_TYPE,
#     gpu_count=GPU_COUNT,
#     container_disk_gb=CONTAINER_DISK_GB,
# )
#
# # Display results
# print("\n" + "="*70)
# if result['success']:
#     print("✅ AUTOMATED WORKFLOW COMPLETED SUCCESSFULLY")
#     print(f"\nProcessed: {result['samples_processed']} samples")
#     print(f"Duration: {result['duration_seconds']:.1f}s ({result['duration_seconds']/60:.1f} minutes)")
#     print(f"Output file: {result['output_file']}")
#     print(f"Session logs: {result['session_dir']}")
#     
#     # Load and preview results
#     with open(result['output_file']) as f:
#         outputs = [json.loads(line) for line in f]
#     
#     print(f"\nFirst output sample:")
#     print(json.dumps(outputs[0], indent=2)[:500] + "...")
#     
# else:
#     print("❌ AUTOMATED WORKFLOW FAILED")
#     print(f"Error: {result['error']}")
#     print(f"Duration: {result['duration_seconds']:.1f}s")
#     print(f"Session logs: {result['session_dir']}")
# print("="*70)

print("(Cell ready - uncomment code above to run automated workflow)")

## ⚙️ Advanced Configuration

### Error Handling Options

```python
result = judge_with_automated_workflow(
    ...,
    cleanup_on_error=False,  # Keep pod running on error for debugging
)
```

**Warning**: If `cleanup_on_error=False`, you must manually terminate the pod to avoid ongoing charges!

### Custom GPU Configuration

```python
result = judge_with_automated_workflow(
    ...,
    gpu_type="NVIDIA RTX A6000",  # Use cheaper GPU
    gpu_count=1,
    container_disk_gb=100,  # Reduce disk size
)
```

### Batch Processing Multiple Models

```python
models_to_judge = [
    "anthropic__claude-sonnet-4.5",
    "openai__gpt-4o",
    "google__gemini-2.0",
]

results = []
for model_name in models_to_judge:
    print(f"\n{'='*70}")
    print(f"Processing: {model_name}")
    print(f"{'='*70}")
    
    # Load samples for this model
    samples = load_samples_from_phase4(model_name)
    
    # Run automated workflow
    result = judge_with_automated_workflow(
        task_name=TASK_NAME,
        model_name=model_name,
        input_samples=samples,
        judge_model=JUDGE_MODEL,
        worker_script=POD_WORKER_SCRIPT_VERBOSE,
        log_dir=LOG_DIR,
    )
    
    results.append(result)
    
    if not result['success']:
        print(f"⚠️  Model {model_name} failed, continuing...")

# Summary
print(f"\n{'='*70}")
print(f"BATCH PROCESSING COMPLETE")
print(f"{'='*70}")
for i, (model, result) in enumerate(zip(models_to_judge, results)):
    status = "✅" if result['success'] else "❌"
    print(f"{status} {model}: {result['samples_processed']} samples")
```

### Resume Partial Runs

If a run fails partway through, you can resume by:

1. Check which samples were already processed
2. Filter those out from your input
3. Run automation on remaining samples
4. Merge the output files

```python
# Load partial output
with open('output_partial.jsonl') as f:
    completed = {s['osq_id'] for s in (json.loads(line) for line in f)}

# Filter remaining samples
remaining = [s for s in all_samples if s['osq_id'] not in completed]
print(f"Resuming: {len(remaining)} samples remaining")

# Run on remaining samples
result = judge_with_automated_workflow(..., input_samples=remaining)
```

## 📊 Automated vs Manual Workflow

| Aspect | Automated Workflow | Manual Debugging Workflow |
|--------|-------------------|---------------------------|
| **Execution** | Single function call | 24 individual cells |
| **Use Case** | Production runs | Debugging & learning |
| **Error Recovery** | Automatic retries | Manual intervention |
| **Progress Tracking** | Real-time streaming | Step-by-step observation |
| **Cleanup** | Automatic | Manual |
| **Logging** | Comprehensive file logs | Console output |
| **Batch Processing** | Easy (loop over models) | Tedious |
| **Learning Curve** | Low (copy example) | High (understand each step) |
| **Flexibility** | Limited (preset stages) | High (modify any step) |
| **Cost Risk** | Low (auto-termination) | Medium (manual cleanup) |

### When to Use Each

**Use Automated Workflow when:**
- Running production evaluations
- Processing large batches (100+ samples)
- You trust the worker script
- You want hands-off execution
- Cost optimization is important

**Use Manual Workflow when:**
- Debugging connection issues
- Developing new worker scripts
- Learning how the system works
- Need fine-grained control
- Investigating pod environment issues

**Pro tip**: Start with manual workflow to understand the system, then switch to automated workflow for production runs.

# Manual Debugging Workflow

This section walks through the LLM judge workflow **step-by-step** for debugging purposes.

## Workflow Overview
1. **Pod Creation** - Create and verify a single RunPod instance
2. **SSH Connection** - Test SSH connectivity and basic commands
3. **Pod Provisioning** - Install dependencies (Ollama, models, etc.)
4. **Worker Script Upload** - Deploy and verify the Python worker script
5. **Test Execution** - Run a small test batch
6. **Results Verification** - Download and validate outputs
7. **Cleanup** - Terminate pod and clean up resources

**Each cell is independent** - you can re-run any step as needed for debugging.


**Prerequisites:** This workflow requires that Cell 3 (Configuration) has been run first to set up constants like `IMAGE_NAME`, `GPU_TYPE`, `JUDGE_MODEL`, etc.

**Worker Script Version:** This workflow uses `POD_WORKER_SCRIPT_VERBOSE` (Version 3) for debugging.


## Step 1: Initialize Session Variables

Set up session-level variables that will persist across cells.


In [20]:
# Note: time, json, Path, datetime imported from Cell 3 (Configuration)

# Session variables (will persist across cells)
session = {
    'pod_id': None,
    'ssh': None,
    'ssh_host': None,
    'ssh_port': None,
    'log_file': None,
    'start_time': time.time()
}

# Create session log
session_log_dir = Path(LOG_DIR) / f"debug_session_{int(time.time())}"
session_log_dir.mkdir(parents=True, exist_ok=True)
session['log_file'] = session_log_dir / 'session.log'

# def log(msg, level='INFO'):
#     """Session logger with file and console output"""
#     timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#     formatted = f"[{timestamp}] [{level}] {msg}"
#     print(formatted)
#     with open(session['log_file'], 'a') as f:
#         f.write(formatted + '\n')

def log(msg, level='INFO'):
    """Session logger with file and console output"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    formatted = f"[{timestamp}] [{level}] {msg}"
    
    print(formatted)

    # Force UTF-8 so characters like ▶ don't break on Windows
    with open(session['log_file'], 'a', encoding='utf-8') as f:
        f.write(formatted + '\n')


log(f"Session initialized. Log file: {session['log_file']}")
log(f"Using judge model: {JUDGE_MODEL}")
log(f"Sample limit (SAMPLE_N): {SAMPLE_N}")
log(f"Models to judge: {len(models_to_judge)}")

print(f"\n{'='*60}")
print(f"SESSION READY")
print(f"{'='*60}")


[2025-12-02 01:49:52] [INFO] Session initialized. Log file: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs\debug_session_1764658192\session.log
[2025-12-02 01:49:52] [INFO] Using judge model: gpt-oss:120b
[2025-12-02 01:49:52] [INFO] Sample limit (SAMPLE_N): 10
[2025-12-02 01:49:52] [INFO] Models to judge: 19

SESSION READY


## Step 2: Create RunPod Instance

Create a single pod with verbose logging. This cell will:
- Request a pod with specified GPU type
- Set up SSH access
- Configure environment variables
- Save pod ID to session


In [21]:
import runpod

log("Creating RunPod instance...")
log(f"  Image: {IMAGE_NAME}")
log(f"  GPU: {GPU_TYPE} x {GPU_COUNT}")
log(f"  Disk: {CONTAINER_DISK_GB} GB")

try:
    pod = create_pod_with_retries(
        log_func=log,
        max_attempts=3,
        retry_delay=10,
        name=f"llm-judge-debug-{int(time.time())}",
        image_name=IMAGE_NAME,
        gpu_type_id=GPU_TYPE,
        gpu_count=GPU_COUNT,
        container_disk_in_gb=CONTAINER_DISK_GB,
        min_vcpu_count=4,
        min_memory_in_gb=16,
        ports="22/tcp,11434/http",
        env={"OLLAMA_HOST": "0.0.0.0"},
        support_public_ip=True,
        start_ssh=True,
    )
    
    session['pod_id'] = pod['id']
    log(f" Pod created successfully!")
    log(f"  Pod ID: {session['pod_id']}")
    
    # Display full pod metadata
    print(f"\n{'='*60}")
    print("POD METADATA:")
    print(f"{'='*60}")
    print(json.dumps(pod, indent=2))
    
except Exception as e:
    log(f"Failed to create pod: {e}", level='ERROR')
    raise


[2025-12-02 01:49:53] [INFO] Creating RunPod instance...
[2025-12-02 01:49:53] [INFO]   Image: runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04
[2025-12-02 01:49:53] [INFO]   GPU: NVIDIA H200 x 1
[2025-12-02 01:49:53] [INFO]   Disk: 200 GB
[2025-12-02 01:49:53] [INFO] Attempt 1/3: creating pod...
raw_response: {'data': {'podFindAndDeployOnDemand': {'id': 'mi6w8ql1iv18qy', 'imageName': 'runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04', 'env': ['OLLAMA_HOST=0.0.0.0', 'PUBLIC_KEY=ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIB0U12zQAIppWu15PQqpNJHZd7+uFrYwuqP6CuH03FI2 ryan.a.bell2.civ@us.navy.mil\n'], 'machineId': 'yp109vaz8l5x', 'machine': {'podHostId': 'mi6w8ql1iv18qy-64410a41'}}}}
[2025-12-02 01:49:53] [INFO] Pod created: mi6w8ql1iv18qy
[2025-12-02 01:49:53] [INFO]  Pod created successfully!
[2025-12-02 01:49:53] [INFO]   Pod ID: mi6w8ql1iv18qy

POD METADATA:
{
  "id": "mi6w8ql1iv18qy",
  "imageName": "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04",

## Step 3: Wait for SSH Port and Extract Connection Details

Poll the pod metadata until SSH port is available.


In [22]:
if not session['pod_id']:
    raise RuntimeError("Pod not created! Run previous cell first.")

log("Waiting for SSH port to become available...")

for attempt in range(60):
    try:
        pod_meta = runpod.get_pod(session['pod_id'])
        runtime = pod_meta.get('runtime', {})
        
        if runtime and runtime.get('ports'):
            for port_entry in runtime['ports']:
                if port_entry.get('privatePort') == 22 and port_entry.get('isIpPublic'):
                    session['ssh_host'] = port_entry['ip']
                    session['ssh_port'] = port_entry['publicPort']
                    log(f"SSH port found!")
                    log(f"  Host: {session['ssh_host']}")
                    log(f"  Port: {session['ssh_port']}")
                    break
        
        if session['ssh_host']:
            break
        
        if attempt % 5 == 0:
            log(f"Attempt {attempt+1}/60: Waiting...")
        
        time.sleep(10)
        
    except Exception as e:
        log(f"Error checking pod status: {e}", level='WARN')
        time.sleep(10)

if not session['ssh_host']:
    raise RuntimeError("SSH port never became available after 10 minutes")

print(f"\n{'='*60}")
print(f"SSH CONNECTION INFO:")
print(f"{'='*60}")
print(f"Host: {session['ssh_host']}")
print(f"Port: {session['ssh_port']}")
print(f"User: root")
print(f"Key:  {SSH_KEY_PATH}")


[2025-12-02 01:50:37] [INFO] Waiting for SSH port to become available...
[2025-12-02 01:50:37] [INFO] SSH port found!
[2025-12-02 01:50:37] [INFO]   Host: 149.7.4.151
[2025-12-02 01:50:37] [INFO]   Port: 10459

SSH CONNECTION INFO:
Host: 149.7.4.151
Port: 10459
User: root
Key:  C:\Users\rabel/.ssh/id_ed25519


## Step 4: Establish SSH Connection

Connect via SSH with detailed error handling.


In [23]:
import paramiko

if not session['ssh_host']:
    raise RuntimeError("No SSH host! Run previous cells first.")

log("Establishing SSH connection...")

try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    log(f"  Loading SSH key from: {SSH_KEY_PATH}")
    private_key = paramiko.Ed25519Key.from_private_key_file(SSH_KEY_PATH)
    
    log(f"  Connecting to {session['ssh_host']}:{session['ssh_port']}...")
    ssh.connect(
        hostname=session['ssh_host'],
        port=session['ssh_port'],
        username='root',
        pkey=private_key,
        timeout=30,
        banner_timeout=30
    )
    
    session['ssh'] = ssh
    log("SSH connection established!")
    
except Exception as e:
    log(f"SSH connection failed: {e}", level='ERROR')
    log(f"Check that the pod is running and SSH is enabled", level='ERROR')
    raise


[2025-12-02 01:50:39] [INFO] Establishing SSH connection...
[2025-12-02 01:50:39] [INFO]   Loading SSH key from: C:\Users\rabel/.ssh/id_ed25519
[2025-12-02 01:50:39] [INFO]   Connecting to 149.7.4.151:10459...
[2025-12-02 01:50:40] [INFO] SSH connection established!


## Step 5: Test Basic SSH Commands

Run simple commands to verify SSH is working properly.


In [24]:
if not session['ssh']:
    raise RuntimeError("SSH not connected! Run previous cell first.")

test_commands = [
    ('whoami', 'Check current user'),
    ('hostname', 'Get hostname'),
    ('pwd', 'Check working directory'),
    ('df -h', 'Check disk space'),
    ('free -h', 'Check memory'),
]

print(f"\n{'='*60}")
print("TESTING BASIC SSH COMMANDS")
print(f"{'='*60}\n")

for cmd, desc in test_commands:
    try:
        log(f"Running: {cmd} ({desc})")
        stdin, stdout, stderr = session['ssh'].exec_command(cmd)
        output = stdout.read().decode().strip()
        error = stderr.read().decode().strip()
        
        if error:
            log(f"stderr: {error}", level='WARN')
        
        print(f"\n[{desc}]")
        print(output)
        
    except Exception as e:
        log(f"Failed: {e}", level='ERROR')

log("Basic SSH tests complete")



TESTING BASIC SSH COMMANDS

[2025-12-02 01:50:42] [INFO] Running: whoami (Check current user)

[Check current user]
root
[2025-12-02 01:50:42] [INFO] Running: hostname (Get hostname)

[Get hostname]
9264a93095e0
[2025-12-02 01:50:42] [INFO] Running: pwd (Check working directory)

[Check working directory]
/root
[2025-12-02 01:50:43] [INFO] Running: df -h (Check disk space)

[Check disk space]
Filesystem      Size  Used Avail Use% Mounted on
overlay         200G   16M  200G   1% /
tmpfs            64M     0   64M   0% /dev
shm             176G     0  176G   0% /dev/shm
/dev/nvme5n1p2  879G   43G  792G   6% /usr/bin/nvidia-smi
/dev/md0         12T  4.2T  7.6T  36% /etc/hosts
tmpfs           303G  8.2M  303G   1% /run/nvidia-persistenced/socket
tmpfs           4.0K  4.0K     0 100% /run/nvidia-ctk-hook
tmpfs           1.5T     0  1.5T   0% /proc/acpi
tmpfs           1.5T     0  1.5T   0% /proc/scsi
tmpfs           1.5T     0  1.5T   0% /sys/firmware
tmpfs           1.5T     0  1.5T   0% 

## Step 6: Check GPU Availability

Verify GPU is detected and accessible.


In [25]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

print(f"\n{'='*60}")
print("GPU CHECK")
print(f"{'='*60}\n")

# Try nvidia-smi first
log("Checking for NVIDIA GPU...")
stdin, stdout, stderr = session['ssh'].exec_command('nvidia-smi || echo "nvidia-smi not found"')
output = stdout.read().decode().strip()
print(output)

# Also try lshw if nvidia-smi fails
if 'not found' in output.lower():
    log("nvidia-smi not available, checking with lshw...")
    stdin, stdout, stderr = session['ssh'].exec_command('lshw -C display 2>/dev/null || echo "lshw not installed"')
    output = stdout.read().decode().strip()
    print(output)



GPU CHECK

[2025-12-02 01:50:46] [INFO] Checking for NVIDIA GPU...
Tue Dec  2 06:50:46 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:18:00.0 Off |                    0 |
| N/A   30C    P0             75W /  700W |       1MiB / 143771MiB |      0%      Default |
|                                         |                        |    

## Step 7: Update apt and Install lshw

First provisioning step - update package manager and install lshw.


In [26]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Updating apt and installing lshw...")

cmd = "apt update && apt install -y lshw"

try:
    run_and_check(session['ssh'], cmd, "Install lshw", log)
    log("lshw installed successfully")
except Exception as e:
    log(f"Failed to install lshw: {e}", level='ERROR')
    raise


[2025-12-02 01:50:48] [INFO] Updating apt and installing lshw...
[2025-12-02 01:50:48] [INFO] ▶ Install lshw
[2025-12-02 01:50:55] [INFO] Get:1 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6008 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [60.9 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3535 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1286 kB]
Get:11 http://archive.ubuntu.com/

## Step 8: Install Ollama

Download and install Ollama using the official install script.


In [27]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Installing Ollama...")
log("  This may take 2-3 minutes...")

cmd = "curl -fsSL https://ollama.com/install.sh | sh"

try:
    run_and_check(session['ssh'], cmd, "Install Ollama", log)
    log("Ollama installed successfully")
    
    # Verify installation
    stdin, stdout, stderr = session['ssh'].exec_command('which ollama')
    ollama_path = stdout.read().decode().strip()
    log(f"  Ollama binary: {ollama_path}")
    
except Exception as e:
    log(f"Failed to install Ollama: {e}", level='ERROR')
    raise


[2025-12-02 01:51:08] [INFO] Installing Ollama...
[2025-12-02 01:51:08] [INFO]   This may take 2-3 minutes...
[2025-12-02 01:51:08] [INFO] ▶ Install Ollama
[2025-12-02 01:51:58] [INFO] WARNING: systemd is not running

[2025-12-02 01:51:58] [INFO] stderr: >>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

[2025-12-02 01:51:58] [INFO] ✔ Install Ollama finished.
[2025-12-02 01:51:58] [INFO] Ollama installed successfully
[2025-12-02 01:51:59] [INFO]   Ollama binary: /usr/local/bin/ollama


## Step 9: Start Ollama Server

Launch Ollama in background mode and verify it's running.


In [29]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Starting Ollama server...")

# Start Ollama in background
cmd_start = "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /root/ollama.log 2>&1 &"
try:
    run_and_check(session['ssh'], cmd_start, "Start Ollama server", log)
    log("  Waiting for server to initialize...")
    time.sleep(10)
    
    # Check if process is running
    stdin, stdout, stderr = session['ssh'].exec_command('ps aux | grep ollama | grep -v grep')
    proc_output = stdout.read().decode().strip()
    
    if proc_output:
        log("Ollama server is running")
        print("\nProcess info:")
        print(proc_output)
    else:
        log("Warning: Ollama process not found", level='WARN')
        
except Exception as e:
    log(f"Failed to start Ollama: {e}", level='ERROR')
    raise


[2025-12-02 01:52:45] [INFO] Starting Ollama server...
[2025-12-02 01:52:45] [INFO] ▶ Start Ollama server
[2025-12-02 01:52:45] [INFO] ✔ Start Ollama server finished.
[2025-12-02 01:52:45] [INFO]   Waiting for server to initialize...
[2025-12-02 01:52:56] [INFO] Ollama server is running

Process info:
root         671  0.2  0.0 2153460 29200 ?       Sl   06:52   0:00 ollama serve


## Step 10: Test Ollama API

Verify Ollama is responding to API requests.


In [30]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Testing Ollama API...")

# Test with curl
cmd = "curl -s http://localhost:11434/api/tags"

try:
    stdin, stdout, stderr = session['ssh'].exec_command(cmd)
    output = stdout.read().decode().strip()
    error = stderr.read().decode().strip()
    
    if error:
        log(f"  stderr: {error}", level='WARN')
    
    print("\nOllama API Response:")
    print(output)
    
    # Try to parse as JSON
    try:
        data = json.loads(output)
        log(f"Ollama API is responding (found {len(data.get('models', []))} models)")
    except:
        log("  Response is not valid JSON - may indicate API is not ready", level='WARN')
        
except Exception as e:
    log(f"Failed to test Ollama API: {e}", level='ERROR')
    raise


[2025-12-02 01:53:08] [INFO] Testing Ollama API...

Ollama API Response:
{"models":[]}
[2025-12-02 01:53:08] [INFO] Ollama API is responding (found 0 models)


## Step 11: Pull Judge Model

Download the judge model. **This will take several minutes** depending on model size.


In [31]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log(f"Pulling judge model: {JUDGE_MODEL}")
log("  This may take 5-10 minutes for large models...")

cmd = f"ollama pull {JUDGE_MODEL}"

print(f"\n{'='*60}")
print(f"DOWNLOADING MODEL: {JUDGE_MODEL}")
print(f"{'='*60}\n")

try:
    # Stream output for model pull to see progress
    stdin, stdout, stderr = session['ssh'].exec_command(cmd)
    
    # Read output line by line
    for line in stdout:
        print(line.strip())
    
    # Check for errors
    error = stderr.read().decode().strip()
    if error:
        log(f"  stderr: {error}", level='WARN')
    
    log(f"Model {JUDGE_MODEL} pulled successfully")
    
except Exception as e:
    log(f"Failed to pull model: {e}", level='ERROR')
    raise


[2025-12-02 01:53:15] [INFO] Pulling judge model: gpt-oss:120b
[2025-12-02 01:53:15] [INFO]   This may take 5-10 minutes for large models...

DOWNLOADING MODEL: gpt-oss:120b

[2025-12-02 01:57:18] [WARN]   stderr: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏  22 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏  81 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 121 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 208 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 298 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   1% ▕                  ▏ 346 MB/ 65 GB                  pulling manife

## Step 12: Verify Model is Available

List installed models to confirm the judge model is ready.


In [32]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Verifying installed models...")

cmd = "ollama list"

try:
    stdin, stdout, stderr = session['ssh'].exec_command(cmd)
    output = stdout.read().decode().strip()
    
    print(f"\n{'='*60}")
    print("INSTALLED MODELS:")
    print(f"{'='*60}\n")
    print(output)
    
    if JUDGE_MODEL in output:
        log(f"Judge model {JUDGE_MODEL} is ready")
    else:
        log(f"Warning: {JUDGE_MODEL} not found in model list", level='WARN')
        
except Exception as e:
    log(f"Failed to list models: {e}", level='ERROR')
    raise


[2025-12-02 01:58:46] [INFO] Verifying installed models...

INSTALLED MODELS:

NAME            ID              SIZE     MODIFIED           
gpt-oss:120b    a951a23b46a1    65 GB    About a minute ago
[2025-12-02 01:58:46] [INFO] Judge model gpt-oss:120b is ready


## Step 13: Upload Worker Script to Pod

Deploy the POD_WORKER_SCRIPT_VERBOSE to /root/pod_worker.py


In [35]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Uploading worker script...")

# Create the upload command using heredoc
upload_cmd = f"cat > /root/pod_worker.py << 'WORKER_EOF'\n{POD_WORKER_SCRIPT_VERBOSE}\nWORKER_EOF"

try:
    run_and_check(session['ssh'], upload_cmd, "Upload worker script", log)
    log("Worker script uploaded")
    
    # Get file size
    stdin, stdout, stderr = session['ssh'].exec_command('wc -l /root/pod_worker.py')
    line_count = stdout.read().decode().strip()
    log(f"  Script size: {line_count}")
    
except Exception as e:
    log(f"Failed to upload worker script: {e}", level='ERROR')
    raise


[2025-12-02 01:59:55] [INFO] Uploading worker script...
[2025-12-02 01:59:55] [INFO] ▶ Upload worker script
[2025-12-02 01:59:56] [INFO] ✔ Upload worker script finished.
[2025-12-02 01:59:56] [INFO] Worker script uploaded
[2025-12-02 01:59:56] [INFO]   Script size: 284 /root/pod_worker.py


## Step 14: Verify Worker Script

Display the first and last lines of the uploaded script to confirm it's correct.


In [36]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Verifying worker script...")

print(f"\n{'='*60}")
print("FIRST 20 LINES:")
print(f"{'='*60}\n")

stdin, stdout, stderr = session['ssh'].exec_command('head -20 /root/pod_worker.py')
print(stdout.read().decode())

print(f"\n{'='*60}")
print("LAST 10 LINES:")
print(f"{'='*60}\n")

stdin, stdout, stderr = session['ssh'].exec_command('tail -10 /root/pod_worker.py')
print(stdout.read().decode())

log("Worker script verification complete")


[2025-12-02 01:59:59] [INFO] Verifying worker script...

FIRST 20 LINES:

import json
import requests
from datetime import datetime
import sys

JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}



LAST 10 LINES:

        input_path=args.input,
        output_path=args.output,
        judge_model=args.judge_model,
        prompt_id=args.prompt_id,
        temperature

## Step 15: Prepare Test Data (Small Sample)

Create a minimal test input with just 2-3 samples for fast debugging.


In [37]:
# Select first model to test with
if not models_to_judge:
    raise RuntimeError("No models to judge! Check the 'Discover Models to Judge' section.")

test_model = models_to_judge[0]
log(f"Using test model: {test_model}")

# Load phase 4 samples for this model
model_dir = PHASE4_ROOT.parent / TASK_NAME / test_model
sample_file = newest(model_dir.glob("samples_*.jsonl"))

log(f"Loading samples from: {sample_file}")
all_samples = load_jsonl(sample_file)

# Use only first 3 samples for testing
test_samples = all_samples[:3]
log(f"Selected {len(test_samples)} samples for testing")

# Create test input file
test_input_file = session_log_dir / 'test_input.jsonl'
with open(test_input_file, 'w') as f:
    for i, sample in enumerate(test_samples):
        f.write(json.dumps({'sample_id': i, 'phase4_row': sample}) + '\n')

log(f"Test input created: {test_input_file}")

# Display first sample for inspection
print(f"\n{'='*60}")
print("SAMPLE 0 (for inspection):")
print(f"{'='*60}\n")
print(json.dumps(test_samples[0], indent=2)[:1000])
print("\n[... truncated ...]\n")


[2025-12-02 02:00:06] [INFO] Using test model: anthropic__claude-sonnet-4.5
[2025-12-02 02:00:06] [INFO] Loading samples from: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\..\phase4_inference\output\sysengbench-osq\anthropic__claude-sonnet-4.5\samples_sysengbench-osq_2025-11-16T21-02-49.453852.jsonl
[2025-12-02 02:00:06] [INFO] Selected 3 samples for testing
[2025-12-02 02:00:06] [INFO] Test input created: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs\debug_session_1764658192\test_input.jsonl

SAMPLE 0 (for inspection):

{
  "doc_id": 0,
  "doc": {
    "Question ID": 1,
    "Tags": "Introduction to risk",
    "INCOSE Handbook Category": "INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures",
    "question": "What best describes the concept of uncertainty in systems engineering?",
    "choiceA": "The process of systematically improving and optimizing a system for efficiency.",
    "choiceB": "The condition wher

## Step 16: Upload Test Data to Pod

Transfer the test input file to the pod.


In [38]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Uploading test data to pod...")

# Create test directory on pod
remote_test_dir = "/root/test_job"
run_and_check(session['ssh'], f"mkdir -p {remote_test_dir}", "Create test dir", log)

# Upload using SFTP
remote_input = f"{remote_test_dir}/input.jsonl"
remote_output = f"{remote_test_dir}/output.jsonl"

sftp = session['ssh'].open_sftp()
try:
    sftp.put(str(test_input_file), remote_input)
    log(f"Uploaded to {remote_input}")
    
    # Verify upload
    stdin, stdout, stderr = session['ssh'].exec_command(f'wc -l {remote_input}')
    line_count = stdout.read().decode().strip()
    log(f"  Remote file: {line_count}")
    
finally:
    sftp.close()


[2025-12-02 02:00:19] [INFO] Uploading test data to pod...
[2025-12-02 02:00:19] [INFO] ▶ Create test dir
[2025-12-02 02:00:19] [INFO] ✔ Create test dir finished.
[2025-12-02 02:00:20] [INFO] Uploaded to /root/test_job/input.jsonl
[2025-12-02 02:00:20] [INFO]   Remote file: 3 /root/test_job/input.jsonl


## Step 17: Run Worker Script with Test Data

Execute the worker script on the pod with **real-time output streaming**.

This is the critical step - watch for any Python errors or inference issues.


In [39]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Running worker script with test data...")
log("  This will stream output in real-time...")

# Build worker command
worker_cmd = (
    f"python3 /root/pod_worker.py "
    f"--input {remote_input} "
    f"--output {remote_output} "
    f"--judge-model {JUDGE_MODEL} "
    f"--prompt-id {PROMPT_ID} "
    f"--temperature {TEMPERATURE} "
    f"--max-tokens {MAX_TOKENS} "
    f"--task-name {TASK_NAME} "
    f"--model-name {test_model}"
)

print(f"\n{'='*60}")
print("WORKER SCRIPT OUTPUT:")
print(f"{'='*60}\n")
print(f"Command: {worker_cmd}\n")
print(f"{'='*60}\n")

# Define streaming helper if not already defined
def stream_ssh_output_debug(ssh, cmd, log_func):
    """Stream SSH command output line-by-line"""
    log_func(f"Executing: {cmd}")
    
    chan = ssh.get_transport().open_session()
    chan.exec_command(cmd)
    
    while True:
        # Read stdout
        if chan.recv_ready():
            data = chan.recv(4096).decode()
            for line in data.splitlines():
                print(f"[pod] {line}")
                log_func(f"[pod] {line}")
        
        # Read stderr
        if chan.recv_stderr_ready():
            data = chan.recv_stderr(4096).decode()
            for line in data.splitlines():
                print(f"[pod:stderr] {line}")
                log_func(f"[pod:stderr] {line}")
        
        # Check if done
        if chan.exit_status_ready():
            exit_code = chan.recv_exit_status()
            log_func(f"Exit code: {exit_code}")
            if exit_code != 0:
                raise RuntimeError(f"Command failed with exit code {exit_code}")
            break
        
        time.sleep(0.2)

try:
    stream_ssh_output_debug(session['ssh'], worker_cmd, log)
    log("Worker script completed successfully")
except Exception as e:
    log(f"Worker script failed: {e}", level='ERROR')
    raise


[2025-12-02 02:00:24] [INFO] Running worker script with test data...
[2025-12-02 02:00:24] [INFO]   This will stream output in real-time...

WORKER SCRIPT OUTPUT:

Command: python3 /root/pod_worker.py --input /root/test_job/input.jsonl --output /root/test_job/output.jsonl --judge-model gpt-oss:120b --prompt-id p1 --temperature 0.0 --max-tokens 2000 --task-name sysengbench-osq --model-name anthropic__claude-sonnet-4.5


[2025-12-02 02:00:24] [INFO] ▶ Executing: python3 /root/pod_worker.py --input /root/test_job/input.jsonl --output /root/test_job/output.jsonl --judge-model gpt-oss:120b --prompt-id p1 --temperature 0.0 --max-tokens 2000 --task-name sysengbench-osq --model-name anthropic__claude-sonnet-4.5
[pod] [worker] Total OSQ samples: 3
[2025-12-02 02:00:25] [INFO] [pod] [worker] Total OSQ samples: 3
[pod] [worker] Progress: 1/3
[2025-12-02 02:00:25] [INFO] [pod] [worker] Progress: 1/3
[pod] [worker] Progress: 2/3
[2025-12-02 02:01:22] [INFO] [pod] [worker] Progress: 2/3
[pod] [worke

## Step 18: Check Worker Script Process

Verify if the Python process is still running (or completed).


In [40]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Checking for running Python processes...")

stdin, stdout, stderr = session['ssh'].exec_command('ps aux | grep python')
output = stdout.read().decode().strip()

print(f"\n{'='*60}")
print("PYTHON PROCESSES:")
print(f"{'='*60}\n")
print(output if output else "No Python processes found")


[2025-12-02 02:01:43] [INFO] Checking for running Python processes...

PYTHON PROCESSES:

root        1546  0.0  0.0   4364  1540 ?        Ss   07:01   0:00 bash -c ps aux | grep python
root        1548  0.0  0.0   3472  1548 ?        S    07:01   0:00 grep python


## Step 19: Check Output File on Pod

Verify the output file was created and contains data.


In [41]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Checking output file...")

# Check if file exists
stdin, stdout, stderr = session['ssh'].exec_command(f'ls -lh {remote_output}')
ls_output = stdout.read().decode().strip()
ls_error = stderr.read().decode().strip()

print(f"\n{'='*60}")
print("OUTPUT FILE INFO:")
print(f"{'='*60}\n")

if ls_error:
    print(f"ERROR: {ls_error}")
    log("✗ Output file not found!", level='ERROR')
else:
    print(ls_output)
    
    # Count lines
    stdin, stdout, stderr = session['ssh'].exec_command(f'wc -l {remote_output}')
    line_count = stdout.read().decode().strip()
    print(f"\nLine count: {line_count}")
    
    log(f"✓ Output file exists: {line_count}")
    
    # Show first few lines
    print(f"\n{'='*60}")
    print("FIRST 3 LINES:")
    print(f"{'='*60}\n")
    stdin, stdout, stderr = session['ssh'].exec_command(f'head -3 {remote_output}')
    print(stdout.read().decode())


[2025-12-02 02:01:51] [INFO] Checking output file...

OUTPUT FILE INFO:

-rw-r--r-- 1 root root 25K Dec  2 07:01 /root/test_job/output.jsonl

Line count: 3 /root/test_job/output.jsonl
[2025-12-02 02:01:52] [INFO] ✓ Output file exists: 3 /root/test_job/output.jsonl

FIRST 3 LINES:

{"sample_id": 0, "phase4_row": {"doc_id": 0, "doc": {"Question ID": 1, "Tags": "Introduction to risk", "INCOSE Handbook Category": "INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures", "question": "What best describes the concept of uncertainty in systems engineering?", "choiceA": "The process of systematically improving and optimizing a system for efficiency.", "choiceB": "The condition where the outcomes of system functions are not predictable due to lack of information or variability.", "choiceC": "A method for analyzing the costs and benefits of a system over its lifecycle.", "choiceD": "The act of integrating different system components into a cohesive whole.", "answer": "B", "lab

## Step 20: Download Output File

Transfer the output file back to local machine.


In [42]:
if not session['ssh']:
    raise RuntimeError("SSH not connected!")

log("Downloading output file...")

local_output = session_log_dir / 'test_output.jsonl'

sftp = session['ssh'].open_sftp()
try:
    sftp.get(remote_output, str(local_output))
    log(f"Downloaded to {local_output}")
    
    # Show file size
    file_size = local_output.stat().st_size
    log(f"  File size: {file_size} bytes")
    
finally:
    sftp.close()


[2025-12-02 02:02:06] [INFO] Downloading output file...
[2025-12-02 02:02:07] [INFO] Downloaded to c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs\debug_session_1764658192\test_output.jsonl
[2025-12-02 02:02:07] [INFO]   File size: 25323 bytes


## Step 21: Validate Output Structure

Parse and inspect the output to ensure it has the expected format.


In [43]:
log("Validating output structure...")

output_records = load_jsonl(local_output)

print(f"\n{'='*60}")
print("OUTPUT VALIDATION:")
print(f"{'='*60}\n")
print(f"Total records: {len(output_records)}")
print(f"Expected records: {len(test_samples)}")

if len(output_records) == len(test_samples):
    log("Record count matches!")
else:
    log(f"Warning: Record count mismatch: {len(output_records)} vs {len(test_samples)}", level='WARN')

# Inspect first record
if output_records:
    print(f"\n{'='*60}")
    print("FIRST OUTPUT RECORD:")
    print(f"{'='*60}\n")
    print(json.dumps(output_records[0], indent=2))
    
    # Check for required fields
    required_fields = ['sample_id', 'judgment', 'judge_model', 'prompt_id']
    missing = [f for f in required_fields if f not in output_records[0]]
    
    if missing:
        log(f"Warning: Missing fields: {missing}", level='WARN')
    else:
        log("All required fields present")
    
    # Check judgment structure
    judgment = output_records[0].get('judgment', {})
    if isinstance(judgment, dict):
        log(f"  Judgment keys: {list(judgment.keys())}")
        if 'overall_score' in judgment:
            log(f"  Overall score: {judgment['overall_score']}")
            log("Judgment structure looks good")
    else:
        log(f"Warning: Judgment is not a dict: {type(judgment)}", level='WARN')
else:
    log("No output records found!", level='ERROR')


[2025-12-02 02:02:48] [INFO] Validating output structure...

OUTPUT VALIDATION:

Total records: 3
Expected records: 3
[2025-12-02 02:02:48] [INFO] Record count matches!

FIRST OUTPUT RECORD:

{
  "sample_id": 0,
  "phase4_row": {
    "doc_id": 0,
    "doc": {
      "Question ID": 1,
      "Tags": "Introduction to risk",
      "INCOSE Handbook Category": "INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures",
      "question": "What best describes the concept of uncertainty in systems engineering?",
      "choiceA": "The process of systematically improving and optimizing a system for efficiency.",
      "choiceB": "The condition where the outcomes of system functions are not predictable due to lack of information or variability.",
      "choiceC": "A method for analyzing the costs and benefits of a system over its lifecycle.",
      "choiceD": "The act of integrating different system components into a cohesive whole.",
      "answer": "B",
      "label": 1,
      "

## Step 22: Close SSH Connection

Gracefully close the SSH connection.


In [45]:
if session['ssh']:
    log("Closing SSH connection...")
    try:
        session['ssh'].close()
        session['ssh'] = None
        log("SSH connection closed")
    except Exception as e:
        log(f"Warning: Error closing SSH: {e}", level='WARN')
else:
    log("SSH already closed")


[2025-12-02 02:07:49] [INFO] Closing SSH connection...
[2025-12-02 02:07:49] [INFO] SSH connection closed


## Step 23: Terminate Pod

**WARNING:** This will terminate the pod and you'll lose all data on it.

Only run this when you're done debugging!


In [46]:
if session['pod_id']:
    log(f"Terminating pod {session['pod_id']}...")
    
    try:
        runpod.terminate_pod(session['pod_id'])
        log("Pod terminated")
        session['pod_id'] = None
        
    except Exception as e:
        log(f"Error terminating pod: {e}", level='WARN')
else:
    log("No pod to terminate")


[2025-12-02 02:07:53] [INFO] Terminating pod mi6w8ql1iv18qy...
[2025-12-02 02:07:53] [INFO] Pod terminated


## Step 24: Session Summary

Display summary of the debugging session.


In [47]:
import time

duration = time.time() - session['start_time']
minutes = int(duration // 60)
seconds = int(duration % 60)

print(f"\n{'='*60}")
print("SESSION SUMMARY")
print(f"{'='*60}\n")
print(f"Duration: {minutes}m {seconds}s")
print(f"Log file: {session['log_file']}")
print(f"Test model: {test_model}")
print(f"Samples processed: {len(output_records) if 'output_records' in locals() else 'N/A'}")
print(f"\nPod terminated: {'Yes' if not session['pod_id'] else 'No (still running!)'}")
print(f"SSH closed: {'Yes' if not session['ssh'] else 'No (still open!)'}")

log(f"Session complete. Total time: {minutes}m {seconds}s")

print(f"\n{'='*60}")
print("NEXT STEPS:")
print(f"{'='*60}")
print("1. Review the log file for any errors or warnings")
print("2. Inspect the output JSONL to verify judgment quality")
print("3. If everything looks good, you can scale up to full automation")
print("4. Increase SAMPLE_N or use the parallel execution notebook")
print(f"\n{'='*60}\n")



SESSION SUMMARY

Duration: 18m 10s
Log file: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs\debug_session_1764658192\session.log
Test model: anthropic__claude-sonnet-4.5
Samples processed: 3

Pod terminated: Yes
SSH closed: Yes
[2025-12-02 02:08:02] [INFO] Session complete. Total time: 18m 10s

NEXT STEPS:
1. Review the log file for any errors or warnings
2. Inspect the output JSONL to verify judgment quality
3. If everything looks good, you can scale up to full automation
4. Increase SAMPLE_N or use the parallel execution notebook


